In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- tools_relocate ---

print("✅ Fixtures loaded")

✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_tools_relocate():
    def relocate(data: pd.DataFrame, x, before=None, after=None):
        x = np.atleast_1d(x)
        columns = data.columns
        columns2 = columns[~columns.isin(x)]
        if before is None and after is None:
            return data[list(x) + list(columns2)]
        if before is None:
            pos = list(columns2).index(after) + 1
        else:
            pos = list(columns2).index(before)
        return data[list(columns2[:pos]) + list(x) + list(columns2[pos:])]
    return relocate

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_tools_relocate():
    import numpy as np

    def relocate(data: pl.DataFrame, x, before=None, after=None):
        x = np.atleast_1d(x).tolist()
        columns = data.columns
        columns2 = [c for c in columns if c not in x]
        if before is None and after is None:
            return data.select(x + columns2)
        if before is None:
            pos = columns2.index(after) + 1
        else:
            pos = columns2.index(before)
        return data.select(columns2[:pos] + x + columns2[pos:])
    return relocate

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: tools_relocate ===

def _cols(obj):
    return list(obj.columns)

def _compare_relocate_columns(before_result, gen_result, label):
    _before_cols = _cols(before_result)
    _gen_cols = _cols(gen_result)
    if _before_cols == _gen_cols:
        print(f"✅ {label}: MATCH columns={_gen_cols}")
    else:
        print(f"❌ {label}: MISMATCH — before={_before_cols}, gen={_gen_cols}")

# L1 smoke – generated
try:
    _r = gen_tools_relocate()
    print("✅ L1 smoke gen_tools_relocate: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_tools_relocate: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_tools_relocate()
    print("✅ L1 smoke before_tools_relocate: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_tools_relocate: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – call the returned relocate function and compare column order.
try:
    _bf = before_tools_relocate()
    _gf = gen_tools_relocate()
    _pd_df = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6], "d": [7, 8]})
    _pl_df = pl.from_pandas(_pd_df)
    _compare_relocate_columns(
        _bf(_pd_df, "c"),
        _gf(_pl_df, "c"),
        "L2 equivalence tools_relocate move to front",
    )
except Exception as _e:
    print(f"❌ L2 equivalence tools_relocate: setup error — {type(_e).__name__}: {_e}")

# L3 branch – execute returned relocate for default, after, before, and multi-column placement.
try:
    _bf = before_tools_relocate()
    _gf = gen_tools_relocate()
    _pd_df = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6], "d": [7, 8]})
    _pl_df = pl.from_pandas(_pd_df)
    for _label, _kwargs in [
        ("front", {"x": "c"}),
        ("after", {"x": ["a", "d"], "after": "b"}),
        ("before", {"x": "d", "before": "b"}),
        ("numpy_x_after", {"x": np.array(["a", "c"]), "after": "d"}),
    ]:
        _before_edge = _bf(_pd_df, **_kwargs)
        _gen_edge = _gf(_pl_df, **_kwargs)
        _compare_relocate_columns(_before_edge, _gen_edge, f"L3 edge tools_relocate {_label}")
except Exception as _e:
    print(f"❌ L3 edge tools_relocate branch placement: {type(_e).__name__}: {_e}")

# L3 edge – single-column data frame should remain stable.
try:
    _bf = before_tools_relocate()
    _gf = gen_tools_relocate()
    _pd_one = pd.DataFrame({"x": [1, 2]})
    _pl_one = pl.from_pandas(_pd_one)
    _compare_relocate_columns(_bf(_pd_one, "x"), _gf(_pl_one, "x"), "L3 edge tools_relocate single column")
except Exception as _e:
    print(f"❌ L3 edge tools_relocate single column: {type(_e).__name__}: {_e}")

# L3 edge – missing anchor should fail on both sides.
try:
    _bf = before_tools_relocate()
    _gf = gen_tools_relocate()
    _pd_df = pd.DataFrame({"a": [1], "b": [2]})
    _pl_df = pl.from_pandas(_pd_df)
    _before_exc = _gen_exc = None
    try:
        _bf(_pd_df, "a", after="missing")
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        _gf(_pl_df, "a", after="missing")
    except Exception as _e:
        _gen_exc = type(_e).__name__
    if _before_exc == _gen_exc == "ValueError":
        print("✅ L3 edge tools_relocate missing anchor: MATCH (ValueError)")
    else:
        print(f"❌ L3 edge tools_relocate missing anchor: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 edge tools_relocate missing anchor setup: {type(_e).__name__}: {_e}")

# L3 edge – missing moved column should be rejected by both sides, even if exception classes differ by backend.
try:
    _bf = before_tools_relocate()
    _gf = gen_tools_relocate()
    _pd_df = pd.DataFrame({"a": [1], "b": [2]})
    _pl_df = pl.from_pandas(_pd_df)
    _before_exc = _gen_exc = None
    try:
        _bf(_pd_df, "missing")
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        _gf(_pl_df, "missing")
    except Exception as _e:
        _gen_exc = type(_e).__name__
    if _before_exc and _gen_exc:
        print(f"✅ L3 edge tools_relocate missing moved column: both rejected (before={_before_exc}, gen={_gen_exc})")
    else:
        print(f"❌ L3 edge tools_relocate missing moved column: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 edge tools_relocate missing moved column setup: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_tools_relocate: OK, type= function
✅ L1 smoke before_tools_relocate: OK
✅ L2 equivalence tools_relocate move to front: MATCH columns=['c', 'a', 'b', 'd']
✅ L3 edge tools_relocate front: MATCH columns=['c', 'a', 'b', 'd']
✅ L3 edge tools_relocate after: MATCH columns=['b', 'a', 'd', 'c']
✅ L3 edge tools_relocate before: MATCH columns=['a', 'd', 'b', 'c']
✅ L3 edge tools_relocate numpy_x_after: MATCH columns=['b', 'd', 'a', 'c']
✅ L3 edge tools_relocate single column: MATCH columns=['x']
✅ L3 edge tools_relocate missing anchor: MATCH (ValueError)
✅ L3 edge tools_relocate missing moved column: both rejected (before=KeyError, gen=ColumnNotFoundError)
